# Data fetching notebook (optional)

Explore the Cricbuzz API and push one completed match into the database.

**Run Jupyter from the project root** (`jupyter notebook` in `C:\\Labmentix\\CricBuzz`) so `config`, `api` and `services` import correctly. Each API call uses quota, and responses are saved to `data/raw/` so you can re-run cells offline.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'config.py').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

from api.client import CricbuzzClient
from services import transformers as tf

client = CricbuzzClient(raw_dir=ROOT / 'data' / 'raw')
print('API key configured:', client.is_configured)

## 1. Recent matches (1 API call)
The response nests matches as `typeMatches -> seriesMatches -> seriesAdWrapper -> matches`, with adverts mixed in.

In [ ]:
recent = client.get_recent_matches()
cards = tf.match_cards(recent)
import pandas as pd
pd.DataFrame([c.__dict__ for c in cards])[['match_id','match_format','team1','team2','status']].head(15)

## 2. One scorecard (1 API call)
Pick a completed Test/ODI/T20I `match_id` from the table above.

In [ ]:
match_id = cards[0].match_id  # change me
raw = client.get_scorecard(match_id)
meta = tf.parse_match_meta(raw)
{k: meta[k] for k in ('match_description','match_format','match_date','status','toss_decision','victory_type','victory_margin')}

In [ ]:
innings = tf.parse_scorecard(raw, meta)
for inn in innings:
    print(inn.innings_no, inn.batting_team, inn.runs, inn.wickets, inn.overs)
pd.DataFrame(innings[0].batting) if innings else 'no innings yet'

## 3. Push it into the database
`IngestionService` writes teams, venue, series, match, players and innings in **one transaction**. Re-running is safe: the match's innings are replaced, never duplicated.

In [ ]:
from services.ingestion_service import IngestionService
rows = IngestionService(client=client).ingest_match(match_id)
print(f'{rows} innings rows written')
print('Quota left:', client.quota)